# Study 259 -- News-Tone
## For the quants: predictive regressions, the within-month placebo, lagged-tone tradability

*Part of [Open-Alpha-Lab](../../../README.md). See the [desk methodology](../../../METHODOLOGY.md).*


## Setup

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))          # the study package
sys.path.insert(0, os.path.abspath("../../.."))    # repo root (quantlab/)
%matplotlib inline
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (9.5, 5.0), "axes.grid": True,
                     "grid.alpha": .3, "axes.spines.top": False, "axes.spines.right": False})
RED, AMBER, GREEN, GREY = "#c0392b", "#dab617", "#2ea44f", "#8b949e"

from news_tone import data, strategy as st

CACHE_PATH = data.GSPC_CACHE
HAVE_REAL = os.path.exists(CACHE_PATH)

if HAVE_REAL:
    df_real = data.build_real_panel(fetch=False)
    print(f"Real panel loaded: {len(df_real)} trading days "
          f"({df_real.index[0].date()} -> {df_real.index[-1].date()})")
else:
    df_real = None
    print("No real cache -- frozen headline numbers from R dict will be used")


Real panel loaded: 4780 trading days (2007-01-03 -> 2025-12-31)


In [2]:

# Frozen headline numbers (mirror of docs/results.md, as-of 2026-06-17)
R = {'n_days': 4780, 'win_start': '2007-01-03', 'win_end': '2025-12-31', 'fp': 'c3d8b303cf23', 'nd_beta': 0.00209, 'nd_t': 5.74, 'nd_r2': 0.0169, 'sd_t': 5.42, 'contemp_corr': 0.125, 'sign_ann': 35.0, 'sign_t': 9.49, 'sign_sr': 1.78, 'sign_hit': 0.554, 'sign_net_t': 9.46, 'sign_net_ann': 34.9, 'tilt_ann': 14.6, 'tilt_t': 6.18, 'tilt_sr': 1.11, 'buyhold_ann': 10.3, 'buyhold_t': 2.65, 'buyhold_sr': 0.52, 'demean_t': 0.0, 'demean_r2': 0.0, 'priormonth_ann': 3.8, 'priormonth_t': 0.97, 'priormonth_sr': 0.19, 'sub1_ann': 33.9, 'sub1_t': 4.11, 'sub2_ann': 25.7, 'sub2_t': 5.69, 'sub3_ann': 43.9, 'sub3_t': 7.0, 'syn_null_t': 0.9, 'syn_null_sr': 0.31, 'syn_g2_t': 7.29, 'syn_g2_sr': 2.59}


## Positive control: the engine finds a signal when one is planted

Before trusting any real-tape number we confirm the machinery is truthful on the
deterministic synthetic tape (`data.synthetic_daily`). The DGP plants a linear
link `ret_{t+1} = gamma * z(tone_t) + noise`. At `gamma = 0` the tone is a
persistent-but-useless AR(1) series (the null); at `gamma > 0` the next-day
predictability is real.

> 💡 **In plain words:** we feed the code data where we *know* the answer. On
> pure-noise tone it must read ~zero; with a planted link it must light up. Only
> then do we believe what it says about the real market.


In [3]:
print("Synthetic control (n_days=2000, seed=259):")
for gamma in [0.0, 0.002, 0.004]:
    d, _ = data.synthetic_daily(n_days=2000, gamma=gamma, seed=259)
    reg = st.predictive_regression(d, target_col="fwd_ret")
    sig = st.tone_sign_signal(d)
    s = st.summarize(st.timing_returns(d, sig))
    tag = "NULL " if gamma == 0 else "planted"
    print(f"  gamma={gamma:.3f} [{tag}]: reg t={reg['tstat']:+.2f}  r2={reg['r2']:.4f}  | sign SR={s['sharpe_ann']:.2f}  t={s['tstat']:+.2f}")
print("\n-> Null reads ~zero (t<1); planted gamma lights up monotonically. Engine is truthful.")


Synthetic control (n_days=2000, seed=259):


  gamma=0.000 [NULL ]: reg t=+1.25  r2=0.0007  | sign SR=0.31  t=+0.90


  gamma=0.002 [planted]: reg t=+9.79  r2=0.0417  | sign SR=2.59  t=+7.29


  gamma=0.004 [planted]: reg t=+18.32  r2=0.1325  | sign SR=4.78  t=+12.46

-> Null reads ~zero (t<1); planted gamma lights up monotonically. Engine is truthful.


## Same-day vs next-day: the mechanical-vs-tradable gap

Two regressions of return on tone z-score. The *same-day* fit (`ret`) is
mechanical -- bad-news days are down days. The *next-day* fit (`fwd_ret`) is the
only one a trader could act on. On the curated proxy *both* look significant,
which is the first red flag: a low-frequency hindsight tone leaks its forward
drift.


In [4]:
if HAVE_REAL:
    sd = st.predictive_regression(df_real, target_col="ret")
    nd = st.predictive_regression(df_real, target_col="fwd_ret")
    print("SAME-DAY  (ret ~ tone):     beta={:+.5f}  HAC t={:+.2f}  r2={:.4f}  n={}".format(sd["beta"], sd["tstat"], sd["r2"], sd["n"]))
    print("NEXT-DAY  (fwd_ret ~ tone): beta={:+.5f}  HAC t={:+.2f}  r2={:.4f}  n={}".format(nd["beta"], nd["tstat"], nd["r2"], nd["n"]))
else:
    print(f"SAME-DAY  HAC t = {R['sd_t']:+.2f}")
    print(f"NEXT-DAY  HAC t = {R['nd_t']:+.2f}  (beta={R['nd_beta']:+.5f}, r2={R['nd_r2']:.4f})")
print("\nBoth significant -> suspicious. A genuine forward signal should NOT")
print("be roughly as strong as the mechanical contemporaneous one.")


SAME-DAY  (ret ~ tone):     beta=+0.00201  HAC t=+5.42  r2=0.0156  n=4779
NEXT-DAY  (fwd_ret ~ tone): beta=+0.00209  HAC t=+5.74  r2=0.0169  n=4780

Both significant -> suspicious. A genuine forward signal should NOT
be roughly as strong as the mechanical contemporaneous one.


## The within-month placebo: where the t-stat goes to die

The tone proxy is constant within each calendar month and was curated *knowing*
each month's outcome. So the regression is really exploiting the month-level
mean. Subtract each month's own mean forward return and re-run: if any genuine
*day-to-day* information existed, the slope would survive. It does not.


In [5]:
if HAVE_REAL:
    d2 = df_real.copy()
    d2["ym"] = d2.index.to_period("M")
    d2["fwd_dm"] = d2["fwd_ret"] - d2.groupby("ym")["fwd_ret"].transform("mean")
    dm = st.predictive_regression(d2[["tone", "fwd_dm"]].rename(columns={"fwd_dm": "fwd_ret"}),
                                  target_col="fwd_ret")
    print(f"Within-month-demeaned next-day reg: beta={dm['beta']:+.8f}  HAC t={dm['tstat']:+.2f}  r2={dm['r2']:.8f}")
else:
    print(f"Within-month-demeaned next-day reg: HAC t = {R['demean_t']:+.2f}  r2 = {R['demean_r2']:.3f}")
print("\n-> EXACTLY ZERO. 100% of the apparent edge was the month-level drift")
print("   that the hindsight-curated proxy already 'knew'. No daily forecast power.")


Within-month-demeaned next-day reg: beta=+0.00000000  HAC t=+0.00  r2=0.00000000

-> EXACTLY ZERO. 100% of the apparent edge was the month-level drift
   that the hindsight-curated proxy already 'knew'. No daily forecast power.


## The honest tradable: lag the tone by a month

A trader on the first day of a month does not know that month's mood. Replace
this month's tone with *last* month's (knowable in advance) and run the sign
strategy. This is the only version that respects the information set.


In [6]:
if HAVE_REAL:
    monthly_lag = data.news_tone_monthly().shift(1)
    keys = df_real.index + pd.offsets.MonthEnd(0)
    d3 = df_real.copy()
    d3["tone_lag"] = monthly_lag.reindex(keys).to_numpy()
    d3 = d3.dropna(subset=["tone_lag", "fwd_ret"])
    g_lag = (np.sign(d3["tone_lag"]) * d3["fwd_ret"]).dropna()
    s_lag = st.summarize(g_lag)
    bh = st.summarize(df_real["fwd_ret"])
    print(f"Prior-month sign-timing : {s_lag['mean_ann']*100:+.1f}%/yr  HAC t={s_lag['tstat']:+.2f}  SR={s_lag['sharpe_ann']:.2f}  hit={s_lag['hit_rate']:.3f}")
    print(f"Buy & hold benchmark    : {bh['mean_ann']*100:+.1f}%/yr  HAC t={bh['tstat']:+.2f}  SR={bh['sharpe_ann']:.2f}")
else:
    print(f"Prior-month sign-timing : {R['priormonth_ann']:+.1f}%/yr  HAC t={R['priormonth_t']:+.2f}  SR={R['priormonth_sr']:.2f}")
    print(f"Buy & hold benchmark    : {R['buyhold_ann']:+.1f}%/yr  HAC t={R['buyhold_t']:+.2f}  SR={R['buyhold_sr']:.2f}")
print("\n-> The honest version is no better than the market's drift. t < 2. No edge.")


Prior-month sign-timing : +3.8%/yr  HAC t=+0.97  SR=0.19  hit=0.508
Buy & hold benchmark    : +10.3%/yr  HAC t=+2.65  SR=0.52

-> The honest version is no better than the market's drift. t < 2. No edge.


## Costs: even the mirage is cost-cheap (because it barely trades)

For completeness, the naive sign-timing strategy nets almost nothing in costs --
the monthly tone flips sign only a handful of times per year, so turnover is
tiny. Cost is *not* what kills this idea; **hindsight** is. We charge a one-way
cost on |position change| x NAV, with short days paying a daily borrow.


In [7]:
if HAVE_REAL:
    sig = st.tone_sign_signal(df_real)
    gross = st.timing_returns(df_real, sig)
    net = st.net_of_costs(sig, gross, one_way_bps=1.0, borrow_bps_per_day=0.04)
    sg, sn = st.summarize(gross), st.summarize(net)
    flips = int(sig.diff().abs().gt(0).sum())
    print(f"Naive sign-timing  gross: {sg['mean_ann']*100:+.1f}%/yr  t={sg['tstat']:+.2f}")
    print(f"Naive sign-timing  net  : {sn['mean_ann']*100:+.1f}%/yr  t={sn['tstat']:+.2f}  (sign flips: {flips})")
    print("\nGross ~ net: turnover is trivial. The +35%/yr is real in-sample -- and")
    print("entirely a hindsight artifact, as the demean and lag tests proved.")
else:
    print(f"Naive sign-timing gross: {R['sign_ann']:+.1f}%/yr t={R['sign_t']:+.2f}")
    print(f"Naive sign-timing net  : {R['sign_net_ann']:+.1f}%/yr t={R['sign_net_t']:+.2f}")


Naive sign-timing  gross: +35.0%/yr  t=+9.49
Naive sign-timing  net  : +34.9%/yr  t=+9.46  (sign flips: 77)

Gross ~ net: turnover is trivial. The +35%/yr is real in-sample -- and
entirely a hindsight artifact, as the demean and lag tests proved.


## Sub-periods of the naive (mirage) strategy: stable *because* it is an artifact

In [8]:
if HAVE_REAL:
    sig = st.tone_sign_signal(df_real)
    gross = st.timing_returns(df_real, sig)
    for a, b in [("2007", "2012"), ("2013", "2018"), ("2019", "2025")]:
        s = st.summarize(gross[a:b])
        print(f"  {a}-{b}: {s['mean_ann']*100:+.1f}%/yr  HAC t={s['tstat']:+.2f}  n={s['n']}")
else:
    print(f"  2007-2012: {R['sub1_ann']:+.1f}%/yr  t={R['sub1_t']:+.2f}")
    print(f"  2013-2018: {R['sub2_ann']:+.1f}%/yr  t={R['sub2_t']:+.2f}")
    print(f"  2019-2025: {R['sub3_ann']:+.1f}%/yr  t={R['sub3_t']:+.2f}")
print("\nUniformly 'significant' across sub-periods -- exactly what a hindsight")
print("artifact looks like. Stability here is a symptom, not a virtue.")


  2007-2012: +33.9%/yr  HAC t=+4.11  n=1510


  2013-2018: +25.7%/yr  HAC t=+5.69  n=1510
  2019-2025: +43.9%/yr  HAC t=+7.00  n=1760

Uniformly 'significant' across sub-periods -- exactly what a hindsight
artifact looks like. Stability here is a symptom, not a virtue.


## Verdict

In [9]:
print("=== Study 259 -- News-Tone ===")
print()
print("Signal: NONE")
print(f"  Naive next-day reg t = +{R['nd_t']:.2f} looks strong, BUT:")
print(f"    - within-month demeaned t = {R['demean_t']:+.2f}  (the apparent edge is 100% month-drift)")
print(f"    - prior-month (honest) sign strat t = {R['priormonth_t']:+.2f}, SR = {R['priormonth_sr']:.2f}  (no edge)")
print("    - tone proxy is hindsight-curated => upper bound, and even that dissolves")
print()
print("Tradability: MIRAGE")
print("  The only 'profitable' version uses information no trader had in real time.")
print("  Honest, information-respecting version is indistinguishable from buy-and-hold.")
print()
print("Proxy/survivorship: NAMED -- curated tone can only OVER-state predictability.")
print()
print("Bottom line: None/Mirage -- aggregate news tone does not move the next day's")
print("  tape in any tradable way. The headline t-stat is a hindsight artifact.")


=== Study 259 -- News-Tone ===

Signal: NONE
  Naive next-day reg t = +5.74 looks strong, BUT:
    - within-month demeaned t = +0.00  (the apparent edge is 100% month-drift)
    - prior-month (honest) sign strat t = +0.97, SR = 0.19  (no edge)
    - tone proxy is hindsight-curated => upper bound, and even that dissolves

Tradability: MIRAGE
  The only 'profitable' version uses information no trader had in real time.
  Honest, information-respecting version is indistinguishable from buy-and-hold.

Proxy/survivorship: NAMED -- curated tone can only OVER-state predictability.

Bottom line: None/Mirage -- aggregate news tone does not move the next day's
  tape in any tradable way. The headline t-stat is a hindsight artifact.
